# Assignment 7: Score-Based Models

This notebook covers **score-based generative models** — a family that learns to estimate the gradient of the log-probability, then uses that gradient to generate samples.

Topics covered (Lecture 7):
- The score function as a direction pointer in probability space
- Score matching: learning scores without the normalising constant
- The manifold hypothesis and why scores fail in low-density regions
- Noise perturbation as the fix: denoising score matching (DSM)
- Langevin dynamics: sampling using only the score function

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

np.random.seed(42)
plt.rcParams['figure.figsize'] = (9, 6)

---
## Part 1: The Score Function

The **score function** of a distribution $p(x)$ is the gradient of the log-probability with respect to $x$:

$$s(x) = \nabla_x \log p(x)$$

Key insight: it **points toward higher probability** and requires no normalisation constant:
$$\nabla_x \log p(x) = \nabla_x \left[\log \tilde{p}(x) - \log Z\right] = \nabla_x \log \tilde{p}(x)$$

For a Gaussian $\mathcal{N}(\mu, \sigma^2)$:
$$\log p(x) = -\tfrac{1}{2}\log(2\pi\sigma^2) - \frac{(x-\mu)^2}{2\sigma^2}$$
$$\Rightarrow \nabla_x \log p(x) = -\frac{x - \mu}{\sigma^2}$$

For a 2D Gaussian $\mathcal{N}(\boldsymbol{\mu}, \boldsymbol{\Sigma})$:
$$\nabla_x \log p(x) = -\boldsymbol{\Sigma}^{-1}(x - \boldsymbol{\mu})$$

In [ ]:
def analytical_score_gaussian_1d(x, mu, sigma):
    """Score function of N(mu, sigma^2): d/dx log p(x).

    log N(x|mu,sigma^2) = -0.5*log(2*pi*sigma^2) - (x-mu)^2 / (2*sigma^2)

    Args:
        x     : float or np.ndarray
        mu    : float
        sigma : float, standard deviation

    Returns:
        score : float or np.ndarray, nabla_x log p(x) = -(x - mu) / sigma^2
    """
    # TODO: return -(x - mu) / sigma**2
    return -(x - mu) / sigma**2


def analytical_score_gaussian_2d(x, mu, Sigma):
    """Score function of a 2D Gaussian N(mu, Sigma).

    nabla_x log p(x) = -Sigma^{-1} (x - mu)

    Args:
        x     : np.ndarray (2,)
        mu    : np.ndarray (2,)
        Sigma : np.ndarray (2, 2), covariance matrix

    Returns:
        score : np.ndarray (2,)
    """
    # TODO:
    # return -np.linalg.solve(Sigma, x - mu)
    # Hint: np.linalg.solve(A, b) is more stable than np.linalg.inv(A) @ b
    return -np.linalg.solve(Sigma, x - mu)

In [ ]:
# Sanity checks for analytical_score_gaussian_1d
print(f'score at x=0.0, mu=0, sigma=1:  {analytical_score_gaussian_1d(0.0, 0.0, 1.0):.4f}  (expected  0.0)')
print(f'score at x=1.0, mu=0, sigma=1:  {analytical_score_gaussian_1d(1.0, 0.0, 1.0):.4f}  (expected -1.0)')
print(f'score at x=-2.0, mu=0, sigma=1: {analytical_score_gaussian_1d(-2.0, 0.0, 1.0):.4f}  (expected  2.0)')

# Sanity check for 2D score
mu_2d = np.array([0.0, 0.0])
Sigma_2d = np.eye(2)
s2d = analytical_score_gaussian_2d(np.array([1.0, -1.0]), mu_2d, Sigma_2d)
print(f'\n2D score at (1,-1), N(0,I): {s2d}  (expected [-1.  1.])')

---
## Part 2: Visualising Score Fields

The score vector at each point shows which direction increases log-probability. Near the mode, all arrows point inward — toward the region of highest density. For a mixture of Gaussians, the score field shows two (or more) attracting basins with arrows pointing toward each mode.

In [ ]:
def analytical_score_mixture_1d(x, pis, mus, sigmas):
    """Score function of a 1D Gaussian mixture p(x) = sum_k pi_k N(x|mu_k, sigma_k^2).

    Uses: nabla_x log p(x) = sum_k [pi_k N(x|mu_k, sigma_k^2) * score_k(x)] / p(x)

    Args:
        x      : np.ndarray (N,)
        pis    : list of float, mixing coefficients (sum to 1)
        mus    : list of float, component means
        sigmas : list of float, component stds

    Returns:
        score  : np.ndarray (N,)
    """
    # TODO:
    # For each component k: compute pi_k * N(x|mu_k, sigma_k) and score_k(x)
    # p(x) = sum_k pi_k * N(x|mu_k, sigma_k)
    # nabla log p(x) = (1/p(x)) * sum_k pi_k * N(x|mu_k, sigma_k) * score_k(x)
    # Hint: use norm.pdf (already imported from scipy.stats)
    weighted_scores = sum(
        pis[k] * norm.pdf(x, mus[k], sigmas[k]) * analytical_score_gaussian_1d(x, mus[k], sigmas[k])
        for k in range(len(pis))
    )
    px = sum(
        pis[k] * norm.pdf(x, mus[k], sigmas[k])
        for k in range(len(pis))
    )
    return weighted_scores / px

In [ ]:
# Plot 1: 1D score field for N(0,1) and bimodal mixture
x_grid = np.linspace(-6, 6, 300)

# Single Gaussian N(0,1)
pdf_single = norm.pdf(x_grid, 0, 1)
score_single = analytical_score_gaussian_1d(x_grid, 0.0, 1.0)

# Bimodal mixture: 0.5*N(-2,0.5) + 0.5*N(2,0.5)
pis_mix = [0.5, 0.5]
mus_mix = [-2.0, 2.0]
sigs_mix = [0.5, 0.5]
pdf_mix = sum(pis_mix[k] * norm.pdf(x_grid, mus_mix[k], sigs_mix[k]) for k in range(2))
score_mix = analytical_score_mixture_1d(x_grid, pis_mix, mus_mix, sigs_mix)

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Row 0: PDFs
axes[0, 0].plot(x_grid, pdf_single, color='tab:blue')
axes[0, 0].fill_between(x_grid, pdf_single, alpha=0.2, color='tab:blue')
axes[0, 0].set_title('PDF: $\\mathcal{N}(0, 1)$')
axes[0, 0].set_xlabel('$x$'); axes[0, 0].set_ylabel('$p(x)$'); axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(x_grid, pdf_mix, color='tab:orange')
axes[0, 1].fill_between(x_grid, pdf_mix, alpha=0.2, color='tab:orange')
axes[0, 1].set_title('PDF: $0.5\\mathcal{N}(-2,0.5^2) + 0.5\\mathcal{N}(2,0.5^2)$')
axes[0, 1].set_xlabel('$x$'); axes[0, 1].set_ylabel('$p(x)$'); axes[0, 1].grid(True, alpha=0.3)

# Row 1: Score functions
axes[1, 0].plot(x_grid, score_single, color='tab:blue')
axes[1, 0].axhline(0, color='k', lw=0.8, ls='--')
axes[1, 0].axvline(0, color='gray', lw=0.8, ls=':')
axes[1, 0].set_title('Score $\\nabla_x \\log p(x)$: $\\mathcal{N}(0,1)$')
axes[1, 0].set_xlabel('$x$'); axes[1, 0].set_ylabel('Score'); axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(x_grid, score_mix, color='tab:orange')
axes[1, 1].axhline(0, color='k', lw=0.8, ls='--')
axes[1, 1].set_title('Score: bimodal mixture')
axes[1, 1].set_xlabel('$x$'); axes[1, 1].set_ylabel('Score'); axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: 2D score field — quiver plot for N(0, [[2,1],[1,1]])
mu_2d_vis = np.array([0.0, 0.0])
Sigma_2d_vis = np.array([[2.0, 1.0], [1.0, 1.0]])

gx = np.linspace(-4, 4, 20)
gy = np.linspace(-4, 4, 20)
GX, GY = np.meshgrid(gx, gy)

U = np.zeros_like(GX)
V = np.zeros_like(GY)
for i in range(GX.shape[0]):
    for j in range(GX.shape[1]):
        pt = np.array([GX[i, j], GY[i, j]])
        s = analytical_score_gaussian_2d(pt, mu_2d_vis, Sigma_2d_vis)
        U[i, j] = s[0]
        V[i, j] = s[1]

# Colour by magnitude
magnitude = np.sqrt(U**2 + V**2)

plt.figure(figsize=(7, 6))
plt.quiver(GX, GY, U, V, magnitude, cmap='viridis', alpha=0.85)
plt.colorbar(label='Score magnitude $\\|s(x)\\|$')
plt.scatter([0], [0], color='red', s=120, zorder=5, label='Mode $\\mu$')
plt.title('2D Score Field: $\\mathcal{N}(\\mathbf{0}, \\Sigma)$\n(arrows point toward the mode)')
plt.xlabel('$x_1$'); plt.ylabel('$x_2$')
plt.legend(); plt.grid(True, alpha=0.3); plt.axis('equal')
plt.tight_layout()
plt.show()

---
## Part 3: Score Matching Objective

We want to train $s_\theta(x) \approx \nabla_x \log p(x)$ without knowing $p(x)$.

**Hyvärinen (2005)** showed the Fisher divergence:
$$J(\theta) = \frac{1}{2} \mathbb{E}_p\!\left[\|s_\theta(x) - \nabla_x \log p(x)\|^2\right]$$
can be rewritten (via integration by parts in 1D) as:
$$J(\theta) = \mathbb{E}_p\!\left[\frac{1}{2} s_\theta(x)^2 + \frac{\partial s_\theta}{\partial x}\right] + \text{const}$$

This avoids needing the true score — we only need samples from $p(x)$!

The derivative $\partial s_\theta / \partial x$ is approximated numerically using a finite difference:
$$\frac{\partial s_\theta}{\partial x} \approx \frac{s_\theta(x + \epsilon) - s_\theta(x - \epsilon)}{2\epsilon}$$

In [ ]:
def score_matching_loss_1d(s_fn, X):
    """Score matching loss (Hyvarinen 2005) for 1D data.

    L(theta) = E_x[ (1/2) s_theta(x)^2 + ds_theta/dx ]

    Approximated as the sample mean over X.

    The derivative ds_theta/dx is estimated numerically:
        ds/dx approx (s(x + eps) - s(x - eps)) / (2 * eps)

    Args:
        s_fn : callable, s_fn(x) -> np.ndarray (same shape as x)
        X    : np.ndarray (N,), data samples

    Returns:
        float, score matching loss
    """
    eps = 1e-3
    # TODO:
    # s_vals = s_fn(X)
    # ds_dx  = (s_fn(X + eps) - s_fn(X - eps)) / (2 * eps)
    # return np.mean(0.5 * s_vals**2 + ds_dx)
    s_vals = s_fn(X)
    ds_dx  = (s_fn(X + eps) - s_fn(X - eps)) / (2 * eps)
    return np.mean(0.5 * s_vals**2 + ds_dx)

In [ ]:
# Verify: for the correctly parameterised score of N(0,1), the loss should be at its minimum at mu=0
np.random.seed(0)
X_gauss = np.random.randn(1000)  # samples from N(0,1)

mu_grid = np.linspace(-2, 2, 100)
losses = [score_matching_loss_1d(lambda x, m=m: analytical_score_gaussian_1d(x, m, 1.0), X_gauss)
          for m in mu_grid]

optimal_mu = mu_grid[np.argmin(losses)]
print(f'Sample mean of data:          {X_gauss.mean():.4f}')
print(f'Optimal mu_hat (grid search): {optimal_mu:.4f}  (should match sample mean)')

plt.figure(figsize=(8, 4))
plt.plot(mu_grid, losses, color='tab:blue')
plt.axvline(optimal_mu, color='tab:red', ls='--', label=f'argmin = {optimal_mu:.3f}')
plt.axvline(X_gauss.mean(), color='tab:green', ls=':', label=f'sample mean = {X_gauss.mean():.3f}')
plt.xlabel('$\\hat{\\mu}$'); plt.ylabel('Score matching loss')
plt.title('Score matching loss vs. $\\hat{\\mu}$: minimum at sample mean')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Part 4: Low-Density Region Failure Mode

Score matching is only accurate where $p(x) > 0$ with high probability. In **low-density regions** (between modes of a bimodal distribution), we have few training samples, so the score estimate is unreliable.

The **manifold hypothesis** makes this worse: real data often lies on a low-dimensional manifold, leaving vast regions of input space almost empty. In those regions, score estimates from finite data are noisy and untrustworthy.

In [ ]:
def demonstrate_low_density_failure(n_samples=300, random_state=0):
    """Show that score matching fails in low-density regions.

    Setup: data = 0.5 * N(-3, 0.5^2) + 0.5 * N(3, 0.5^2)
    The region x in (-1, 1) has almost no data, so the score there is unreliable.

    Args:
        n_samples    : int, number of training samples
        random_state : int
    """
    np.random.seed(random_state)
    # TODO:
    # 1. Generate n_samples from the bimodal mixture 0.5*N(-3,0.5) + 0.5*N(3,0.5)
    #    Hint: half from each component
    # 2. Compute the true analytical score over x_plot = np.linspace(-6, 6, 300)
    #    using analytical_score_mixture_1d
    # 3. Estimate score from data using a KDE-style score (perturbed_score_1d from Part 5)
    #    or simply show the histogram to demonstrate sparsity between modes
    # 4. Plot:
    #    - True PDF of the mixture (left y-axis)
    #    - Data histogram (to show sparsity in low-density region)
    #    - True score function (right y-axis)
    #    - Shade the low-density region x in (-1, 1)

    # Step 1: Generate data
    n_half = n_samples // 2
    X_left  = np.random.randn(n_half) * 0.5 - 3.0
    X_right = np.random.randn(n_samples - n_half) * 0.5 + 3.0
    X_data  = np.concatenate([X_left, X_right])

    # Step 2: True score and PDF
    x_plot = np.linspace(-6, 6, 300)
    pis_  = [0.5, 0.5]; mus_ = [-3.0, 3.0]; sigs_ = [0.5, 0.5]
    true_score = analytical_score_mixture_1d(x_plot, pis_, mus_, sigs_)
    true_pdf   = sum(pis_[k] * norm.pdf(x_plot, mus_[k], sigs_[k]) for k in range(2))

    # Step 3: Plot
    fig, ax1 = plt.subplots(figsize=(11, 5))
    ax2 = ax1.twinx()

    ax1.fill_between(x_plot, true_pdf, alpha=0.15, color='tab:blue', label='True PDF')
    ax1.plot(x_plot, true_pdf, color='tab:blue', lw=2)
    ax1.hist(X_data, bins=40, density=True, alpha=0.4, color='tab:gray', label='Data histogram')
    ax1.axvspan(-1, 1, alpha=0.12, color='red', label='Low-density region')
    ax1.set_xlabel('$x$'); ax1.set_ylabel('Density', color='tab:blue')
    ax1.tick_params(axis='y', labelcolor='tab:blue')

    ax2.plot(x_plot, true_score, color='tab:orange', lw=2.5, label='True score')
    ax2.axhline(0, color='k', lw=0.8, ls='--')
    ax2.set_ylabel('Score $\\nabla_x \\log p(x)$', color='tab:orange')
    ax2.tick_params(axis='y', labelcolor='tab:orange')

    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9)

    n_low = np.sum(np.abs(X_data) < 1)
    ax1.set_title(
        f'Low-density failure: {n_low}/{n_samples} samples in $x \\in (-1, 1)$\n'
        'Score is unreliable in the shaded region (no training signal)'
    )
    plt.tight_layout(); plt.show()
    print(f'Samples in low-density region |x| < 1: {n_low} / {n_samples}')


demonstrate_low_density_failure(n_samples=300)

**Reflection:** Why does the score estimate become unreliable in the region $x \in (-1, 1)$, even if our score network is trained with many gradient steps?

**Your answer**: TODO

---
## Part 5: Noise Perturbation Fix

**Solution**: perturb the data with noise $\sigma$ before estimating the score.

Perturbed distribution:
$$q_\sigma(x) = \int \mathcal{N}(x \mid x_0, \sigma^2)\, p(x_0)\, dx_0$$

The score of $q_\sigma$ at a noisy point $z = x_0 + \sigma\varepsilon$ is:
$$\nabla_z \log q_\sigma(z) \approx -\frac{z - x_0}{\sigma^2} \quad \text{[for small } \sigma\text{]}$$

This fills in low-density regions and makes the score estimable everywhere. Larger $\sigma$ gives a smoother, more globally reliable score but blurs fine structure. Smaller $\sigma$ preserves detail but still leaves gaps. The ideal choice balances coverage and fidelity.

In [ ]:
def perturbed_score_1d(z, X_data, sigma):
    """Approximate score of the perturbed distribution q_sigma(z) via KDE score.

    score approx (1/q_sigma(z)) * sum_i N(z|X_i, sigma^2) * (-(z-X_i)/sigma^2)

    This is the exact score of a KDE with bandwidth sigma.

    Args:
        z      : np.ndarray (M,), query points
        X_data : np.ndarray (N,), training data
        sigma  : float, noise level

    Returns:
        score  : np.ndarray (M,)
    """
    # TODO:
    # Vectorised (M, N) weight matrix:
    # weights = norm.pdf(z[:, None], X_data[None, :], sigma)  -> (M, N)
    # scores_per_pair = -(z[:, None] - X_data[None, :]) / sigma**2  -> (M, N)
    # return (weights * scores_per_pair).sum(axis=1) / weights.sum(axis=1)
    weights         = norm.pdf(z[:, None], X_data[None, :], sigma)          # (M, N)
    scores_per_pair = -(z[:, None] - X_data[None, :]) / sigma**2            # (M, N)
    return (weights * scores_per_pair).sum(axis=1) / weights.sum(axis=1)

In [ ]:
# Sanity check: single data point at origin, query at origin -> score = 0
s_check = perturbed_score_1d(np.array([0.0]), np.array([0.0]), 1.0)
print(f'perturbed_score_1d([0.0], [0.0], 1.0) = {s_check[0]:.6f}  (expected 0.0)')

# Check: if single data point at x=1 and query at x=0, score should be negative
s_check2 = perturbed_score_1d(np.array([0.0]), np.array([1.0]), 0.5)
print(f'perturbed_score_1d([0.0], [1.0], 0.5) = {s_check2[0]:.4f}  (expected > 0, points toward x=1)')

In [ ]:
# Compare true score vs perturbed scores at different sigma levels
np.random.seed(1)
n_bimodal = 500
X_bimodal = np.concatenate([
    np.random.randn(n_bimodal // 2) * 0.5 - 3.0,
    np.random.randn(n_bimodal // 2) * 0.5 + 3.0,
])

x_plot = np.linspace(-6, 6, 300)
pis_b  = [0.5, 0.5]; mus_b = [-3.0, 3.0]; sigs_b = [0.5, 0.5]
true_score_b = analytical_score_mixture_1d(x_plot, pis_b, mus_b, sigs_b)

sigmas_to_try = [0.1, 0.5, 1.0, 2.0]
colors_sigma  = ['tab:green', 'tab:orange', 'tab:red', 'tab:purple']

plt.figure(figsize=(11, 5))
plt.plot(x_plot, true_score_b, 'k-', lw=2.5, label='True score', zorder=5)
for sigma, col in zip(sigmas_to_try, colors_sigma):
    pert = perturbed_score_1d(x_plot, X_bimodal, sigma)
    plt.plot(x_plot, pert, color=col, lw=1.5, alpha=0.85, label=f'$\\sigma={sigma}$')

plt.axvspan(-1, 1, alpha=0.08, color='red')
plt.axhline(0, color='k', lw=0.7, ls='--')
plt.ylim(-30, 30)
plt.xlabel('$x$'); plt.ylabel('Score')
plt.title('Effect of noise level $\\sigma$ on KDE score estimate\n'
          '(red band = low-density region; larger $\\sigma$ fills it in)')
plt.legend(fontsize=9); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## Part 6: Denoising Score Matching (PyTorch)

**Denoising Score Matching (DSM)** directly trains a score network to denoise:

$$\mathcal{L}_{\text{DSM}}(\theta, \sigma) = \mathbb{E}_{x \sim p,\, \varepsilon \sim \mathcal{N}(0,I)} \left[\|s_\theta(x + \sigma\varepsilon,\, \sigma) - (-\varepsilon/\sigma)\|^2\right]$$

This equals the score matching objective up to a constant. The target $-\varepsilon/\sigma = -(z - x)/\sigma^2$ where $z = x + \sigma\varepsilon$ is the exact score of the Gaussian kernel $\mathcal{N}(z \mid x, \sigma^2)$.

Key advantages over standard score matching:
- **No Jacobian trace** computation required (avoids $\mathcal{O}(D^2)$ cost)
- **Fills low-density regions** via the noise perturbation
- **Scalable** to high-dimensional data

In [ ]:
import torch
import torch.nn as nn

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
class ScoreNet(nn.Module):
    """Score network s_theta(x, sigma) for 1D/2D data.

    Architecture:
        Concatenate x (D-dim) and log(sigma) (1-dim)
        -> Linear(D+1, H) -> Tanh -> Linear(H, H) -> Tanh -> Linear(H, D)

    Output: score estimate s_theta(x, sigma) approx nabla_x log q_sigma(x)
    """

    def __init__(self, data_dim, hidden_dim=128):
        super().__init__()
        # TODO:
        # self.net = nn.Sequential(
        #     nn.Linear(data_dim + 1, hidden_dim),
        #     nn.Tanh(),
        #     nn.Linear(hidden_dim, hidden_dim),
        #     nn.Tanh(),
        #     nn.Linear(hidden_dim, data_dim),
        # )
        raise NotImplementedError

    def forward(self, x, sigma):
        """Forward pass.

        Args:
            x     : Tensor (B, D)
            sigma : float or Tensor broadcastable to (B, 1), noise level

        Returns:
            score : Tensor (B, D)
        """
        # TODO:
        # If sigma is a scalar, broadcast to (B, 1); else reshape to (B, 1)
        # log_sigma = torch.log(sigma * torch.ones(x.shape[0], 1, device=x.device))
        # x_in = torch.cat([x, log_sigma], dim=1)
        # return self.net(x_in)
        raise NotImplementedError

In [ ]:
# Sanity check: ScoreNet should map (B, D) -> (B, D)
# Complete the __init__ and forward stubs above first.
torch.manual_seed(0)
try:
    _net_test = ScoreNet(data_dim=2, hidden_dim=64).to(device)
    _x_test   = torch.randn(8, 2).to(device)
    _out_test = _net_test(_x_test, sigma=0.3)
    print(f'ScoreNet output shape: {_out_test.shape}  (expected torch.Size([8, 2]))')
    assert _out_test.shape == (8, 2)
    print('ScoreNet forward pass OK.')
except NotImplementedError:
    print('NotImplementedError: implement ScoreNet.__init__ and forward first.')

In [ ]:
def dsm_loss(model, X_batch, sigma):
    """Denoising score matching loss for a single noise level sigma.

    L = E_{x~data, eps~N(0,I)} [ || s_theta(x + sigma*eps, sigma) - (-eps/sigma) ||^2 ]

    Args:
        model   : ScoreNet
        X_batch : Tensor (B, D)
        sigma   : float, noise level

    Returns:
        loss : scalar Tensor
    """
    # TODO:
    # 1. eps    = torch.randn_like(X_batch)
    # 2. z      = X_batch + sigma * eps           (noisy samples)
    # 3. target = -eps / sigma                    (true score at z given x)
    # 4. s_hat  = model(z, sigma)                 (predicted score)
    # 5. return ((s_hat - target)**2).sum(dim=1).mean()
    raise NotImplementedError

In [ ]:
def train_score_net(X_data, sigma=0.3, n_epochs=300, lr=1e-3, batch_size=256):
    """Train a score network using DSM on 2D data.

    Args:
        X_data     : np.ndarray (N, 2)
        sigma      : float, noise level
        n_epochs   : int
        lr         : float
        batch_size : int

    Returns:
        model        : trained ScoreNet
        loss_history : list of float
    """
    X_t = torch.tensor(X_data, dtype=torch.float32).to(device)
    model = ScoreNet(data_dim=X_data.shape[1]).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_t),
        batch_size=batch_size, shuffle=True
    )
    loss_history = []
    for epoch in range(n_epochs):
        epoch_loss = 0.0
        for (xb,) in loader:
            # TODO:
            # opt.zero_grad()
            # loss = dsm_loss(model, xb, sigma)
            # loss.backward()
            # opt.step()
            # epoch_loss += loss.item()
            raise NotImplementedError
        loss_history.append(epoch_loss / len(loader))
        if (epoch + 1) % 100 == 0:
            print(f'Epoch {epoch+1}  loss={loss_history[-1]:.4f}')
    return model, loss_history

In [ ]:
# Generate 2D bimodal training data: 0.5*N([-2,-2], 0.5^2 I) + 0.5*N([2,2], 0.5^2 I)
np.random.seed(42)
N_2d = 2000
X_2d = np.vstack([
    np.random.randn(N_2d // 2, 2) * 0.5 + np.array([-2.0, -2.0]),
    np.random.randn(N_2d // 2, 2) * 0.5 + np.array([ 2.0,  2.0]),
]).astype(np.float32)

plt.figure(figsize=(5, 5))
plt.scatter(X_2d[:, 0], X_2d[:, 1], s=6, alpha=0.4, color='tab:blue')
plt.title('Training data: 2D bimodal Gaussian')
plt.xlabel('$x_1$'); plt.ylabel('$x_2$'); plt.axis('equal'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Train the score network — implement ScoreNet, dsm_loss, and the training loop first!
torch.manual_seed(42)
np.random.seed(42)

try:
    model_trained, loss_hist = train_score_net(X_2d, sigma=0.3, n_epochs=300, lr=1e-3)

    plt.figure(figsize=(8, 4))
    plt.plot(loss_hist, color='tab:blue')
    plt.xlabel('Epoch'); plt.ylabel('DSM loss')
    plt.title('Denoising score matching training loss')
    plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
except NotImplementedError:
    print('NotImplementedError: implement ScoreNet, dsm_loss, and train_score_net first.')

In [ ]:
# Visualise learned score field vs true score field
try:
    model_trained.eval()

    gx2 = np.linspace(-5, 5, 20)
    gy2 = np.linspace(-5, 5, 20)
    GX2, GY2 = np.meshgrid(gx2, gy2)
    pts = np.stack([GX2.ravel(), GY2.ravel()], axis=1).astype(np.float32)

    # True analytical score (mixture of two Gaussians)
    U_true = np.zeros(len(pts)); V_true = np.zeros(len(pts))
    Sigma_comp = np.eye(2) * 0.25  # 0.5^2
    for i, p in enumerate(pts):
        # Weighted combination of two Gaussian scores
        w1 = norm.pdf(p[0], -2, 0.5) * norm.pdf(p[1], -2, 0.5)
        w2 = norm.pdf(p[0],  2, 0.5) * norm.pdf(p[1],  2, 0.5)
        s1 = analytical_score_gaussian_2d(p, np.array([-2., -2.]), Sigma_comp)
        s2 = analytical_score_gaussian_2d(p, np.array([ 2.,  2.]), Sigma_comp)
        s  = (w1 * s1 + w2 * s2) / (w1 + w2 + 1e-30)
        U_true[i] = s[0]; V_true[i] = s[1]

    # Learned score
    with torch.no_grad():
        pts_t  = torch.tensor(pts).to(device)
        s_pred = model_trained(pts_t, sigma=0.3).cpu().numpy()
    U_pred = s_pred[:, 0]; V_pred = s_pred[:, 1]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, U, V, title in zip(
        axes,
        [U_true, U_pred],
        [V_true, V_pred],
        ['True score field', 'Learned score field (DSM)']
    ):
        mag = np.sqrt(U**2 + V**2)
        ax.quiver(GX2, GY2, U.reshape(GX2.shape), V.reshape(GY2.shape),
                  mag.reshape(GX2.shape), cmap='viridis', alpha=0.85)
        ax.scatter(X_2d[:, 0], X_2d[:, 1], s=4, alpha=0.2, color='tab:blue')
        ax.set_title(title); ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
        ax.set_xlim(-5, 5); ax.set_ylim(-5, 5); ax.set_aspect('equal')
        ax.grid(True, alpha=0.2)

    plt.suptitle('Score fields: true vs. learned via DSM')
    plt.tight_layout(); plt.show()
except NotImplementedError:
    print('NotImplementedError: train the model first.')
except NameError:
    print('NameError: train the model first (run the training cell).')

---
## Part 7: Langevin Dynamics Sampling

**Langevin dynamics** iteratively refines a sample using only the score:

$$x_{k+1} = x_k + \frac{\varepsilon}{2}\, s_\theta(x_k) + \sqrt{\varepsilon}\, z, \qquad z \sim \mathcal{N}(0, I)$$

This performs gradient ascent on $\log p(x)$ (moving toward high probability) with added Gaussian noise to prevent collapse and ensure proper exploration. As $\varepsilon \to 0$ and $k \to \infty$, the samples converge to the true distribution $p(x)$.

In [ ]:
def langevin_sample(score_fn, x_init, step_size, n_steps):
    """Run Langevin dynamics to sample from p(x) given s_theta(x).

    x_{k+1} = x_k + (step_size/2) * score_fn(x_k) + sqrt(step_size) * N(0,I)

    Args:
        score_fn  : callable, score_fn(x) -> np.ndarray (same shape as x)
        x_init    : np.ndarray, starting point(s), shape (D,) or (N, D)
        step_size : float, epsilon
        n_steps   : int, number of Langevin steps

    Returns:
        x          : np.ndarray, final samples (same shape as x_init)
        trajectory : list of np.ndarray, one entry per step
    """
    x = x_init.copy().astype(float)
    trajectory = [x.copy()]
    # TODO:
    # For each step t in range(n_steps):
    #   z = np.random.randn(*x.shape)
    #   x = x + (step_size / 2) * score_fn(x) + np.sqrt(step_size) * z
    #   trajectory.append(x.copy())
    # return x, trajectory
    for _ in range(n_steps):
        z = np.random.randn(*x.shape)
        x = x + (step_size / 2) * score_fn(x) + np.sqrt(step_size) * z
        trajectory.append(x.copy())
    return x, trajectory

In [ ]:
# Demo 1: 1D Langevin dynamics starting from x=5.0, target N(0,1)
np.random.seed(7)

score_1d_std = lambda x: analytical_score_gaussian_1d(x, mu=0.0, sigma=1.0)
x_final_1d, traj_1d = langevin_sample(
    score_fn=score_1d_std,
    x_init=np.array(5.0),
    step_size=0.05,
    n_steps=500
)

traj_arr = np.array(traj_1d)  # (n_steps+1,)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(traj_arr, color='tab:blue', lw=0.8, alpha=0.8)
axes[0].axhline(0, color='red', ls='--', lw=1.5, label='Mode $\\mu=0$')
axes[0].set_xlabel('Langevin step'); axes[0].set_ylabel('$x$')
axes[0].set_title('Langevin trajectory: $x_0=5$, target $\\mathcal{N}(0,1)$')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].hist(traj_arr[100:], bins=40, density=True, color='tab:blue', alpha=0.6, label='Langevin samples')
x_ref = np.linspace(-4, 4, 200)
axes[1].plot(x_ref, norm.pdf(x_ref, 0, 1), 'r-', lw=2, label='$\\mathcal{N}(0,1)$')
axes[1].set_xlabel('$x$'); axes[1].set_ylabel('Density')
axes[1].set_title('Sample histogram vs. true PDF (after burn-in)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print(f'Final x: {x_final_1d:.4f}  (expected near 0)')

In [ ]:
# Demo 2: 2D Langevin dynamics using trained ScoreNet
# 200 random starting points -> run Langevin -> samples should cluster around data modes
try:
    model_trained.eval()
    np.random.seed(3)

    def score_fn_2d_net(x_np):
        """Wrap trained ScoreNet as a numpy score function."""
        with torch.no_grad():
            x_t = torch.tensor(x_np, dtype=torch.float32).to(device)
            if x_t.ndim == 1:
                x_t = x_t.unsqueeze(0)
            s = model_trained(x_t, sigma=0.3).cpu().numpy()
            if x_np.ndim == 1:
                return s[0]
            return s

    # Random initializations drawn from N(0, 4I)
    x_inits = np.random.randn(200, 2) * 2.0
    x_final_2d, _ = langevin_sample(
        score_fn=score_fn_2d_net,
        x_init=x_inits,
        step_size=0.01,
        n_steps=500
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].scatter(x_inits[:, 0], x_inits[:, 1], s=20, alpha=0.6, color='tab:gray', label='Init $x_0$')
    axes[0].scatter(X_2d[:, 0], X_2d[:, 1], s=5, alpha=0.2, color='tab:blue', label='Training data')
    axes[0].set_title('Starting points (random)')
    axes[0].set_xlim(-6, 6); axes[0].set_ylim(-6, 6)
    axes[0].set_aspect('equal'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

    axes[1].scatter(x_final_2d[:, 0], x_final_2d[:, 1], s=20, alpha=0.7, color='tab:orange', label='Langevin samples')
    axes[1].scatter(X_2d[:, 0], X_2d[:, 1], s=5, alpha=0.2, color='tab:blue', label='Training data')
    axes[1].set_title('After 500 Langevin steps')
    axes[1].set_xlim(-6, 6); axes[1].set_ylim(-6, 6)
    axes[1].set_aspect('equal'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.suptitle('Langevin dynamics: random init → data manifold')
    plt.tight_layout(); plt.show()

except NotImplementedError:
    print('NotImplementedError: train the score network first (Parts 6).')
except NameError:
    print('NameError: train the score network first.')

**Reflection:** The manifold hypothesis states that real data lies on a low-dimensional manifold. Why does this make standard score matching unreliable?

**Your answer**: TODO

**Reflection:** DSM avoids computing the Jacobian trace (required by standard score matching). Explain why this makes DSM practical for high-dimensional data.

**Your answer**: TODO

---
## Summary

In this notebook you:

- Derived and implemented the score function for Gaussian and mixture distributions
- Visualised score fields as vector fields in 1D and 2D
- Implemented the Hyvärinen score matching objective and verified it recovers the correct parameters
- Demonstrated the low-density region failure mode of standard score matching
- Fixed it with noise perturbation and implemented the KDE score estimate
- Trained a neural score network using denoising score matching (DSM) on 2D data
- Sampled from the learned distribution using Langevin dynamics

| Component | Role |
|---|---|
| `analytical_score_gaussian_1d` | Closed-form score of 1D Gaussian |
| `analytical_score_gaussian_2d` | Closed-form score of 2D Gaussian |
| `analytical_score_mixture_1d` | Score of 1D Gaussian mixture via Bayes rule |
| `score_matching_loss_1d` | Hyvärinen score matching objective |
| `demonstrate_low_density_failure` | Illustrates failure in sparse regions |
| `perturbed_score_1d` | KDE-based score of perturbed distribution |
| `ScoreNet` | Neural score network conditioned on $\sigma$ |
| `dsm_loss` | Denoising score matching training objective |
| `train_score_net` | DSM training loop |
| `langevin_sample` | Langevin MCMC sampler using learned scores |

**Next**: Assignment 8 covers Normalising Flows — an alternative family of generative models with exact likelihood computation.